In [ ]:
!pip install SPARQLWrapper
!pip install sparql-dataframe

In [ ]:
import sparql_dataframe

endpoint = "http://localhost:7200/repositories/mqtt4ssn"

In [ ]:
NS = {
    "mqtt:": "https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#",
    "mqv:": "https://www.w3.org/2019/wot/mqtt#",
    "rdfs:": "http://www.w3.org/2000/01/rdf-schema#",
    "ex:": "http://example.org/",
    "sosa": "http://www.w3.org/ns/sosa/" 
}

In [ ]:
import pandas as pd

def strip_namespaces(df, ns_map=NS):
    def shrink(val):
        if not isinstance(val, str):
            return val
        # ersetze bekannte Namespaces durch Prefix
        for pref, base in ns_map.items():
            if val.startswith(base):
                return pref + val[len(base):]
        # fallback: nur local name nach # oder /
        if "#" in val:
            return val.rsplit("#", 1)[-1]doernern
        if "/" in val:
            return val.rsplit("/", 1)[-1]
        return val

    return df.map(shrink)


## Network Infrastructure

### CQ5.01u - Which Network Participants are exists?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?participant ?type WHERE {
  ?participant a mqtt:NetworkParticipant ;
               a ?type .
  ?type rdfs:subClassOf mqtt:NetworkParticipant .

  FILTER(?type IN (mqtt:Client, mqtt:Broker)) .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short


### CQ5.02u - Which MQTT Clients are connected to which Broker?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?client ?broker WHERE {
  ?client a mqtt:Client ;
          mqtt:isConnectedToBroker ?broker .
  ?broker a mqtt:Broker .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short


### CQ5.03u - Which MQTT Broker has which connected Client?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?broker ?client WHERE {
  ?broker a mqtt:Broker ;
          mqtt:hasConnectedClient ?client .
  ?client a mqtt:Client .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short


###  CQ5.04u - Which MQTT participants use which MQTT Network Connections?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?participant ?type ?connection WHERE {
  ?participant a ?type ;
               mqtt:usesConnection ?connection .
  FILTER(?type IN (mqtt:Client, mqtt:Broker)) .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ5.05u - Which MQTT NetworkConnections exist, are they TLS-secured, and who initiated/accepted them?


In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?connection ?tls ?client ?broker WHERE {
  ?connection a mqtt:NetworkConnection .
  OPTIONAL { ?connection mqtt:usesTLS ?tls }
  OPTIONAL { ?connection mqtt:isInitiatedBy ?client }
  OPTIONAL { ?broker mqtt:isAcceptedBy ?connection }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ5.06u - Which MQTT Clients exists under which ClientID?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?client ?clientID WHERE {
  ?client a mqtt:Client .
  OPTIONAL { ?client mqtt:hasClientID ?clientID }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ5.07u - Which broker exists under which host and port configuration?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?broker ?host ?port WHERE {
  ?broker a mqtt:Broker ;
          OPTIONAL { ?broker mqtt:hasHostAddress ?host }
          OPTIONAL { ?broker mqtt:hasPortNumber ?port }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ5.08u - Which Control Packets are send by which Network Participant?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?participant ?type ?packet WHERE {
  ?participant a ?type ;
             mqtt:sendsControlPacket ?packet .
  ?type rdfs:subClassOf mqtt:NetworkParticipant .
  FILTER(?type IN (mqtt:Client, mqtt:Broker)) .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ5.09u - Which Control Packets are send between both Network Participants?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?client ?broker ?packet WHERE {
  ?client a mqtt:Client ;
        mqtt:sendsControlPacket ?packet .
  ?broker a mqtt:Broker ;
        mqtt:sendsControlPacket ?packet .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ5.10u - Under which Network Connectiom, which Session is established?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?session ?connection WHERE {
  ?session a mqtt:Session ;
        mqtt:establishedByNetworkConnection ?connection .
  ?connection a mqtt:NetworkConnection ;
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ5.11u - Which Session establishes which Network Connection?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?session ?connection WHERE {
  ?connection a mqtt:Connection ;
        mqtt:establishesSession ?session .
  ?session a mqtt:Session ;
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ5.12u - Which MQTT Clients send which MQTT Control Packets?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?client ?packet ?type WHERE {
  ?client a mqtt:Client ;
          mqtt:sendsControlPacket ?packet .

  ?packet a mqtt:ControlPacket ;
          a ?type .

  ?type rdfs:subClassOf* mqtt:ControlPacket .

  FILTER( STRSTARTS(STR(?type), STR(mqtt:)) )
  FILTER( ?type != mqtt:ControlPacket )
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ5.13u - Which MQTT Control Packets were sent (specializations)?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?client ?packet ?type WHERE {
  ?client a mqtt:Client ;
          mqtt:sendsControlPacket ?packet .
  ?packet a mqtt:ControlPacket ;
          a ?type .

  ?type rdfs:subClassOf* mqtt:ControlPacket .

  FILTER( STRSTARTS(STR(?type), STR(mqtt:)) )
  FILTER( ?type != mqtt:ControlPacket )
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ5.14u - Which MQTT Brokers sends which MQTT Control Packets?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?broker ?packet ?type WHERE {
  ?broker a mqtt:Broker ;
          mqtt:sendsControlPacket ?packet .
  ?packet a mqtt:ControlPacket ;
          a ?type .

  ?type rdfs:subClassOf* mqtt:ControlPacket .

  FILTER( STRSTARTS(STR(?type), STR(mqtt:)) )
  FILTER( ?type != mqtt:ControlPacket )
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ5.15u - Which MQTT Control Packets were sent by which Client (inverse)?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?client ?packet ?type WHERE {
  ?client a mqtt:Client ;
          mqtt:sendsControlPacket ?packet .
  ?packet a mqtt:ControlPacket ;
          a ?type .

  ?type rdfs:subClassOf* mqtt:ControlPacket .

  FILTER( STRSTARTS(STR(?type), STR(mqtt:)) )
  FILTER( ?type != mqtt:ControlPacket )
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short